# Kitchen Merged Labels — Comprehensive Data Visualization

**Dataset:** `kitchen_merged_labels_cleaned_oven1_activity_oven2_idle_or_viceversa.csv`  
**Sensors:** Temperature, Humidity, PM2.5, VOC, CO₂  
**Labels:** oven1_label (41 unique activities), oven2_label (33 unique activities)

---

## 0 · Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.preprocessing import MinMaxScaler

matplotlib.rcParams.update({'figure.dpi': 110, 'font.size': 9})
%matplotlib inline

print(f'numpy   {np.__version__}')
print(f'pandas  {pd.__version__}')
print(f'matplotlib {matplotlib.__version__}')

## 1 · Load & Inspect Data

In [ ]:
CSV_PATH = r'd:\nit_internship\Csv_plotting\kitchen_merged_labels_cleaned_oven1_activity_oven2_idle_or_viceversa.csv'

df = pd.read_csv(CSV_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d-%m-%Y %H:%M', errors='coerce')
df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)

print('Shape :', df.shape)
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
df.describe()

In [ ]:
SENSORS = ['data_temperature', 'data_humidity', 'data_pm2_5', 'data_voc', 'data_co2']
SENSOR_LABELS = {
    'data_temperature': 'Temperature (°C)',
    'data_humidity':    'Humidity (%)',
    'data_pm2_5':       'PM2.5 (µg/m³)',
    'data_voc':         'VOC',
    'data_co2':         'CO2 (ppm)',
}
COLORS = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261']

print('Oven-1 unique labels:', df['oven1_label'].nunique())
print('Oven-2 unique labels:', df['oven2_label'].nunique())

## 2 · Time-Series of All Sensors

In [ ]:
fig, axes = plt.subplots(len(SENSORS), 1, figsize=(16, 12), sharex=True)
fig.suptitle('Sensor Time-Series — Kitchen Merged Labels', fontsize=13, fontweight='bold')

for ax, sensor, color in zip(axes, SENSORS, COLORS):
    ax.plot(df['timestamp'], df[sensor], color=color, linewidth=0.7, alpha=0.85)
    ax.set_ylabel(SENSOR_LABELS[sensor], fontsize=8)
    ax.fill_between(df['timestamp'], df[sensor], alpha=0.12, color=color)
    ax.grid(True, linestyle='--', alpha=0.4)

axes[-1].set_xlabel('Timestamp', fontsize=9)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig1_time_series.png', bbox_inches='tight')
plt.show()

## 3 · Sensor Distributions — Violin + Box

In [ ]:
fig, axes = plt.subplots(1, len(SENSORS), figsize=(18, 5))
fig.suptitle('Sensor Value Distributions (Violin + Box)', fontsize=13, fontweight='bold')

for ax, sensor, color in zip(axes, SENSORS, COLORS):
    data = df[sensor].dropna().values
    vp = ax.violinplot(data, showmeans=False, showmedians=False, showextrema=False)
    for body in vp['bodies']:
        body.set_facecolor(color)
        body.set_alpha(0.55)
    bp = ax.boxplot(data, widths=0.18, patch_artist=True,
                    medianprops=dict(color='black', linewidth=1.8))
    bp['boxes'][0].set_facecolor(color + '88')
    ax.set_title(SENSOR_LABELS[sensor], fontsize=8, fontweight='bold')
    ax.set_xticks([])
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig2_distributions.png', bbox_inches='tight')
plt.show()

## 4 · Mean Sensor Values per Oven-1 Label (Top-15 by PM2.5)

In [ ]:
top15_oven1 = (
    df.groupby('oven1_label')[SENSORS]
    .mean()
    .sort_values('data_pm2_5', ascending=False)
    .head(15)
)

fig, axes = plt.subplots(len(SENSORS), 1, figsize=(16, 14))
fig.suptitle('Mean Sensor Values per Oven-1 Activity Label (Top-15 by PM2.5)',
             fontsize=12, fontweight='bold')

for ax, sensor, color in zip(axes, SENSORS, COLORS):
    bars = ax.barh(top15_oven1.index, top15_oven1[sensor],
                   color=color, alpha=0.82, edgecolor='white')
    ax.set_xlabel(SENSOR_LABELS[sensor], fontsize=8)
    ax.invert_yaxis()
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)
    for bar in bars:
        w = bar.get_width()
        ax.text(w * 1.005, bar.get_y() + bar.get_height() / 2,
                f'{w:.1f}', va='center', fontsize=6.5)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig3_oven1_label_means.png', bbox_inches='tight')
plt.show()

## 5 · Mean Sensor Values per Oven-2 Label (Top-15 by PM2.5)

In [ ]:
top15_oven2 = (
    df.groupby('oven2_label')[SENSORS]
    .mean()
    .sort_values('data_pm2_5', ascending=False)
    .head(15)
)

fig, axes = plt.subplots(len(SENSORS), 1, figsize=(16, 14))
fig.suptitle('Mean Sensor Values per Oven-2 Activity Label (Top-15 by PM2.5)',
             fontsize=12, fontweight='bold')

for ax, sensor, color in zip(axes, SENSORS, COLORS):
    bars = ax.barh(top15_oven2.index, top15_oven2[sensor],
                   color=color, alpha=0.82, edgecolor='white')
    ax.set_xlabel(SENSOR_LABELS[sensor], fontsize=8)
    ax.invert_yaxis()
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)
    for bar in bars:
        w = bar.get_width()
        ax.text(w * 1.005, bar.get_y() + bar.get_height() / 2,
                f'{w:.1f}', va='center', fontsize=6.5)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig4_oven2_label_means.png', bbox_inches='tight')
plt.show()

## 6 · Sensor Correlation Heatmap

In [ ]:
corr = df[SENSORS].corr()

fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle('Sensor Correlation Heatmap', fontsize=12, fontweight='bold')

im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
labels = [SENSOR_LABELS[s] for s in SENSORS]
ax.set_xticks(range(len(SENSORS)))
ax.set_yticks(range(len(SENSORS)))
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)

for i in range(len(SENSORS)):
    for j in range(len(SENSORS)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}",
                ha='center', va='center', fontsize=8,
                color='black' if abs(corr.values[i, j]) < 0.7 else 'white')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig5_correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 7 · Pair Scatter Matrix

In [ ]:
n = len(SENSORS)
fig, axes = plt.subplots(n, n, figsize=(16, 14))
fig.suptitle('Sensor Pair Scatter Matrix', fontsize=12, fontweight='bold')

for i, s1 in enumerate(SENSORS):
    for j, s2 in enumerate(SENSORS):
        ax = axes[i][j]
        if i == j:
            ax.hist(df[s1].dropna(), bins=30, color=COLORS[i], alpha=0.75, edgecolor='none')
        else:
            ax.scatter(df[s2], df[s1], s=2, alpha=0.25, color=COLORS[i])
        if j == 0:
            ax.set_ylabel(SENSOR_LABELS[s1], fontsize=6)
        if i == n - 1:
            ax.set_xlabel(SENSOR_LABELS[s2], fontsize=6)
        ax.tick_params(labelsize=5)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig6_pair_scatter.png', bbox_inches='tight')
plt.show()

## 8 · Label Distribution — Oven-1 Bar & Oven-2 Pie

In [ ]:
oven1_counts = df['oven1_label'].value_counts()
oven2_counts = df['oven2_label'].value_counts()
cmap = plt.get_cmap('tab20')

fig, (ax_bar, ax_pie) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Label Distribution — Oven-1 (bar) & Oven-2 (pie)', fontsize=12, fontweight='bold')

# Bar — Oven-1
ax_bar.barh(
    oven1_counts.index, oven1_counts.values,
    color=[cmap(i / len(oven1_counts)) for i in range(len(oven1_counts))],
    edgecolor='white'
)
ax_bar.set_xlabel('Count')
ax_bar.set_title('Oven-1 Label Frequency')
ax_bar.invert_yaxis()
ax_bar.grid(True, axis='x', linestyle='--', alpha=0.4)

# Pie — Oven-2 (group small slices)
threshold = 0.03 * oven2_counts.sum()
oven2_major = oven2_counts[oven2_counts >= threshold].copy()
other_sum = oven2_counts[oven2_counts < threshold].sum()
if other_sum > 0:
    oven2_major['Other'] = other_sum

wedges, texts, autotexts = ax_pie.pie(
    oven2_major.values, labels=oven2_major.index,
    autopct='%1.1f%%', startangle=140,
    colors=[cmap(i / len(oven2_major)) for i in range(len(oven2_major))],
    pctdistance=0.82
)
for t in texts:      t.set_fontsize(7)
for a in autotexts:  a.set_fontsize(6.5)
ax_pie.set_title('Oven-2 Label Share')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig7_label_distribution.png', bbox_inches='tight')
plt.show()

## 9 · Sensor Variation by Cooking Method (Oven-1)

In [ ]:
def cooking_method(label: str) -> str:
    label_lower = label.lower()
    for method in ('boil', 'fry', 'curry', 'reheat', 'idle'):
        if method in label_lower:
            return method.capitalize()
    return 'Other'

df['method_oven1'] = df['oven1_label'].apply(cooking_method)
METHODS = ['Boil', 'Fry', 'Curry', 'Reheat', 'Idle', 'Other']
method_colors = {
    'Boil': '#4CC9F0', 'Fry': '#F72585', 'Curry': '#7209B7',
    'Reheat': '#FB8500', 'Idle': '#8ECAE6', 'Other': '#ADB5BD'
}

fig, axes = plt.subplots(len(SENSORS), 1, figsize=(14, 13))
fig.suptitle('Sensor Variation by Cooking Method (Oven-1)', fontsize=12, fontweight='bold')

for ax, sensor in zip(axes, SENSORS):
    data_by_method = [df.loc[df['method_oven1'] == m, sensor].dropna().values for m in METHODS]
    bp = ax.boxplot(
        data_by_method, tick_labels=METHODS, patch_artist=True,
        medianprops=dict(color='black', linewidth=1.5),
        flierprops=dict(marker='.', markersize=2, alpha=0.4)
    )
    for patch, m in zip(bp['boxes'], METHODS):
        patch.set_facecolor(method_colors[m])
        patch.set_alpha(0.75)
    ax.set_ylabel(SENSOR_LABELS[sensor], fontsize=8)
    ax.grid(True, axis='y', linestyle='--', alpha=0.35)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig8_method_boxplot.png', bbox_inches='tight')
plt.show()

## 10 · Rolling Mean ± Std — PM2.5 & CO₂

In [ ]:
WINDOW = 20

fig, (ax_pm, ax_co2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
fig.suptitle(f'Rolling Mean \u00b1 1 Std (window={WINDOW}) \u2014 PM2.5 & CO2',
             fontsize=12, fontweight='bold')

for ax, sensor, color, label in [
    (ax_pm,  'data_pm2_5', '#2A9D8F', 'PM2.5 (µg/m³)'),
    (ax_co2, 'data_co2',   '#E63946', 'CO2 (ppm)'),
]:
    rm = df[sensor].rolling(WINDOW).mean()
    rs = df[sensor].rolling(WINDOW).std()
    ax.plot(df['timestamp'], rm, color=color, linewidth=1.2, label='Rolling Mean')
    ax.fill_between(df['timestamp'], rm - rs, rm + rs, color=color, alpha=0.2, label='±1 Std')
    ax.plot(df['timestamp'], df[sensor], color=color, linewidth=0.4, alpha=0.35, label='Raw')
    ax.set_ylabel(label, fontsize=8)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.35)

ax_co2.set_xlabel('Timestamp', fontsize=9)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig9_rolling_stats.png', bbox_inches='tight')
plt.show()

## 11 · Normalised Stacked Area — All Sensors

In [ ]:
scaler = MinMaxScaler()
df_norm = pd.DataFrame(scaler.fit_transform(df[SENSORS]), columns=SENSORS)
df_norm['timestamp'] = df['timestamp'].values

fig, ax = plt.subplots(figsize=(16, 5))
fig.suptitle('Normalised Sensor Trends (Min-Max Scaled) \u2014 Stacked Area',
             fontsize=12, fontweight='bold')

ax.stackplot(
    df_norm['timestamp'],
    [df_norm[s] for s in SENSORS],
    labels=[SENSOR_LABELS[s] for s in SENSORS],
    colors=COLORS, alpha=0.72
)
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.set_xlabel('Timestamp', fontsize=9)
ax.set_ylabel('Normalised Value (stacked)', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(r'd:\nit_internship\Csv_plotting\fig10_stacked_area.png', bbox_inches='tight')
plt.show()

---
**All plots rendered and saved.**